# EDA: NYC Airbnb short-term rental prices

This notebook explores the `sample.csv` artifact produced by the `download` step of the pipeline and
decides which basic cleaning operations the `basic_cleaning` step should apply.

Outline:
1. Fetch `sample.csv:latest` from W&B and load it.
2. Profile the raw data.
3. Inspect missing values (and decide **not** to impute them here).
4. Apply the basic fixes: drop price outliers and convert `last_review` to a datetime.
5. Re-profile the cleaned data to verify the fixes.
6. Finish the W&B run.

In [1]:
import os
import numpy as np
import pandas as pd
import wandb
from ydata_profiling import ProfileReport

/opt/conda/envs/nyc_airbnb_dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_9876/351414842.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 1. Load the dataset from W&B

We start a W&B run in the `nyc_airbnb` project (group `eda`) so that this exploration is tracked, and we
fetch the `sample.csv:latest` artifact. Using `use_artifact` also records in W&B that this notebook
consumed that specific artifact version, which keeps the lineage of the pipeline complete.

In [ ]:
# Start a tracked W&B run and fetch the latest version of the raw sample
run = wandb.init(project='nyc_airbnb', group='eda', save_code=True)
local_path = wandb.use_artifact('sample.csv:latest').file()
df_sample = pd.read_csv(local_path)
df_sample.shape

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/codespace/.netrc.
wandb: Currently logged in as: dancoder (dancoder-) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


(20000, 16)

wandb: WARNING Artifact "source-nyc_airbnb-_workspaces_uda-mlops_02_building_a_reproducible_model_workflow_project_EDA.ipynb" already exists with the same content. No new version will be created.


## 2. Profile the raw data

`ydata-profiling` gives an overview of every column: distributions, missing values, correlations and
warnings. The report is saved as HTML in `reports/` so it can be opened in a browser.

In [3]:
profile = ProfileReport(df_sample, title="Pandas Profiling Report - Raw Sample DataFrame - NYC Airbnb")

In [4]:
# Make sure the output folder exists, then save the raw-data report
os.makedirs("./reports", exist_ok=True)
profile.to_file("./reports/eda_sample_input_profiling_report.html")

Summarize dataset:   0%|          | 0/21 [00:00<?, ?it/s, Describe variable: name]


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 67.97it/s]


## 3. Inspect missing values

Before cleaning, we check which columns have missing values and how many.

In [5]:
# Number and percentage of missing values per column (only columns with at least one missing value)
missing = df_sample.isnull().sum()
missing_pct = (missing / len(df_sample) * 100).round(2)
pd.DataFrame({"missing": missing, "missing_%": missing_pct}).query("missing > 0").sort_values("missing", ascending=False)

,missing,missing_%
reviews_per_month,4123,20.62
last_review,4123,20.62
host_name,8,0.04
name,7,0.03


**Observations**

- `last_review` and `reviews_per_month` have the same number of missing values: they are missing for
  listings that have never received a review.
- `name` and `host_name` have a few missing values.

**Decision: we do not impute missing values in this EDA or in the `basic_cleaning` step.**

Missing values will also appear in the new data the model sees in production. If we imputed them here,
the imputation would live only in the training data preparation and would not be applied at inference
time, creating training/serving skew. Instead, imputation is deferred to the **inference pipeline**
(the scikit-learn `Pipeline` built in `train_random_forest`), where the imputers are fitted on the
training data and exported together with the model, so the exact same transformation is applied to
training, validation, test and production data.

## 4. Apply basic fixes

From the profile we apply only two fixes, which the `basic_cleaning` step will reproduce:

1. **Price outliers.** The `price` column contains extreme values (very cheap and very expensive
   listings) that are not representative of the properties we want to model. We keep only listings with
   `10 <= price <= 350` dollars. `Series.between` is inclusive on both ends by default, so listings at
   exactly $10 or $350 are kept.
2. **`last_review` type.** The column is read as text; we convert it to a datetime so that dates can be
   compared and turned into features (e.g. days since the last review). Missing dates stay `NaT`; they are
   not imputed (see section 3).

In [6]:
# Accepted nightly price range (inclusive)
min_price = 10
max_price = 350

# Keep only listings inside the accepted price range
idx = df_sample['price'].between(min_price, max_price)

# Convert last_review from text to datetime; missing values become NaT (no imputation)
df_clean = (
    df_sample[idx]
    .copy()
    .assign(last_review=lambda df: pd.to_datetime(df['last_review']))
)

## 5. Verify the fixes

We check that the price range is respected, that `last_review` is now a datetime, and that the missing
values are still present (not imputed). Then we re-run the profile on the cleaned data.

In [7]:
# Sanity checks on the cleaned data
assert df_clean['price'].between(min_price, max_price).all()
assert pd.api.types.is_datetime64_any_dtype(df_clean['last_review'])
print(f"Rows before: {len(df_sample)}, rows after: {len(df_clean)}")
df_clean.isnull().sum()[lambda s: s > 0]

Rows before: 20000, rows after: 19001


name                    7
host_name               8
last_review          3758
reviews_per_month    3758
dtype: int64

In [8]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  number_

In [9]:
df_clean.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,1.900100e+04,1.900100e+04,19001.000000,19001.000000,19001.000000,19001.000000,19001.000000,15243,15243.000000,19001.000000,19001.000000
mean,1.883041e+07,6.639459e+07,40.728063,-73.950827,122.340456,6.906900,23.797747,2018-10-01 08:24:05.384766720,1.380928,6.583811,109.725383
min,2.539000e+03,2.571000e+03,40.508730,-74.239140,10.000000,1.000000,0.000000,2011-05-12 00:00:00,0.010000,1.000000,0.000000
25%,9.355498e+06,7.728754e+06,40.688820,-73.982050,66.000000,1.000000,1.000000,2018-06-29 12:00:00,0.190000,1.000000,0.000000
50%,1.938754e+07,3.048785e+07,40.721710,-73.954630,100.000000,2.000000,6.000000,2019-05-19 00:00:00,0.720000,1.000000,39.000000
75%,2.891952e+07,1.048354e+08,40.763210,-73.934490,160.000000,5.000000,24.000000,2019-06-23 00:00:00,2.010000,2.000000,219.000000
max,3.648561e+07,2.742733e+08,40.913060,-73.717950,350.000000,1250.000000,607.000000,2019-07-08 00:00:00,27.950000,327.000000,365.000000
std,1.096986e+07,7.782663e+07,0.055389,0.046825,71.530346,21.456544,45.493455,NaN,1.689988,31.154750,130.599899


In [10]:
profile = ProfileReport(df_clean, title="Pandas Profiling Report - Cleaned DataFrame - NYC Airbnb")

In [11]:
# Save the cleaned-data report for comparison with the raw one
profile.to_file("./reports/eda_cleaned_df_profiling_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 55.07it/s]


## 6. Finish the W&B run

Closing the run uploads the notebook code and marks the run as finished in W&B.

In [12]:
run.finish()